## Import Library & Setup Awal

In [ ]:
# =========================================================
# Standard Library
# =========================================================
import glob
import math
import os
import random
import shutil
import time

# =========================================================
# Third-Party Library
# =========================================================
import kagglehub
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from PIL import Image, ImageFilter
from tqdm.notebook import tqdm

# =========================================================
# Machine Learning & Deep Learning
# =========================================================
import tensorflow as tf
from tensorflow.keras import Sequential, layers
from tensorflow.keras.callbacks import (
    Callback,
    CSVLogger,
    EarlyStopping,
    ModelCheckpoint,
    ReduceLROnPlateau,
    TensorBoard
)
from tensorflow.keras.layers import (
    Activation,
    Add,
    BatchNormalization,
    Concatenate,
    Conv2D,
    Dense,
    Dropout,
    Flatten,
    GlobalAveragePooling2D,
    GlobalMaxPooling2D,
    Input,
    MaxPool2D
)
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.metrics import AUC
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.regularizers import l2
from tensorflow.keras.utils import load_img, img_to_array

# =========================================================
# Evaluation Metrics
# =========================================================
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_curve,
    mean_absolute_error
)

In [ ]:
# Setup path / direktori
DATA_DIR = "/kaggle/working/data"
MODEL_DIR = "/kaggle/working/model"

# Buat folder / direktori
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

# Random control (reproducible)
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

## Load Data

### Import Data dari Kaggle

In [ ]:
!kaggle kernels output remonggkircop/final-dataset-capstone -p {DATA_DIR}
!unzip -q {DATA_DIR}/deepfake_clean_final.zip -d {DATA_DIR}/Dataset

In [ ]:
!rm -rf {DATA_DIR}/deepfake_clean_final.zip

### Path Direktori Kelas

In [ ]:
# Ambil path direktori Dataset
dataset_dir = os.path.join(DATA_DIR, "Dataset")
print("Dataset path:", dataset_dir)

# Lihat isi folder dalam Dataset
print("Daftar kelas:", os.listdir(dataset_dir))

In [ ]:
train_dir = os.path.join(dataset_dir, "Train")
val_dir   = os.path.join(dataset_dir, "Validation")
test_dir  = os.path.join(dataset_dir, "Test")

print("Dataset path:", dataset_dir)
print("Train path:", train_dir)
print("Validation path:", val_dir)
print("Test path:", test_dir)

## Preprocessing Data

### Image Generator

In [ ]:
tar_size = (160, 160)
bat_size = 64

train_gen = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    image_size=tar_size,
    batch_size=bat_size,
    label_mode='binary',
    shuffle=True
)

train_pred_gen = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    image_size=tar_size,
    batch_size=bat_size,
    label_mode='binary',
    shuffle=False
)

val_gen = tf.keras.utils.image_dataset_from_directory(
    val_dir,
    image_size=tar_size,
    batch_size=bat_size,
    label_mode='binary',
    shuffle=False
)

test_gen = tf.keras.utils.image_dataset_from_directory(
    test_dir,
    image_size=tar_size,
    batch_size=bat_size,
    label_mode='binary',
    shuffle=False
)

In [ ]:
# Menampilkan jumlah kelas yang ada dalam setiap dataset
train_class = train_gen.class_names
train_pred_class = train_pred_gen.class_names
val_class = val_gen.class_names
test_class = test_gen.class_names

print(train_class)
print(train_pred_class)
print(val_class)
print(test_class)

### Prefetch

In [ ]:
# Buat layer rescale
scaler = tf.keras.layers.Rescaling(1./255)

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

train_gen = train_gen.map(lambda x, y: (scaler(x), y), num_parallel_calls=AUTOTUNE)
train_pred_gen = train_pred_gen.map(lambda x, y: (scaler(x), y), num_parallel_calls=AUTOTUNE)
val_gen = val_gen.map(lambda x, y: (scaler(x), y), num_parallel_calls=AUTOTUNE)
test_gen = test_gen.map(lambda x, y: (scaler(x), y), num_parallel_calls=AUTOTUNE)

### Konfigurasi Layer Preproessing

In [ ]:
# layer augmentasi yang akan digunakan oleh model
data_augmentation = Sequential([
    layers.RandomRotation(0.05, seed=SEED),
    layers.RandomZoom(0.05, seed=SEED),
    layers.RandomFlip("horizontal", seed=SEED),
    layers.RandomGaussianBlur(0.05, seed=SEED)
])

In [ ]:
# Cek Gambar Hasil Augmentasi
images, labels = next(iter(train_gen))
augmented_images = data_augmentation(images)
num_images = 16

cols = 4
rows = math.ceil(num_images / cols)

plt.figure(figsize=(15, 12))

for i in range(num_images):
    plt.subplot(rows, cols, i + 1)

    plt.imshow(augmented_images[i].numpy())
    plt.title(f"Label: {labels[i].numpy()}")

    plt.axis("off")

plt.tight_layout()
plt.show()

## Modeling

### Tensorboard

In [ ]:
tensorboard_dir = f"{MODEL_DIR}/tensorboard_logs"
os.makedirs(tensorboard_dir, exist_ok=True)

train_writer = tf.summary.create_file_writer(f"{tensorboard_dir}/train")
val_writer = tf.summary.create_file_writer(f"{tensorboard_dir}/validation")

In [ ]:
# Akses TensorBoard
%load_ext tensorboard
%tensorboard --logdir {tensorboard_dir}

### Konfigurasi Callbacks

In [ ]:
# Membuat Custom Callbacks : Menghitung Waktu Training
class TrainingProgressCallback(Callback):
    def __init__(self):
        super().__init__()

    def on_train_begin(self, logs=None):
        self.start_time = time.time()

        total_batches = self.params.get("steps", "Unknown")
        total_epochs = self.params.get("epochs", "Unknown")

        print("\nTraining dimulai")
        print(f"Total epoch: {total_epochs}")
        print(f"Total batch per epoch: {total_batches}\n")

    def on_train_end(self, logs=None):
        total_time = time.time() - self.start_time
        final_loss = logs.get("loss", "N/A") if logs else "N/A"
        final_acc = logs.get("accuracy", "N/A") if logs else "N/A"

        print("\nTraining selesai")
        print(f"Waktu training: {total_time:.2f} detik")
        print(f"Final Loss: {final_loss}")
        print(f"Final Accuracy: {final_acc}\n")

In [ ]:
# Membuat folder penyimpanan
model_prog_dir = f"{MODEL_DIR}/model_progress"
os.makedirs(model_prog_dir, exist_ok=True)

# Menyusun callbacks
callbacks = [
    TrainingProgressCallback(),

    CSVLogger(
        f"{model_prog_dir}/training_log.csv",
        append=True
    )
]

### Membangun Arsitektur Model

In [ ]:
# Custom Block (Kumpulan Layer)
def residual_block(x, filters, downsample=False):
    shortcut = x
    stride = 2 if downsample else 1

    # Conv 1
    x = Conv2D(filters, (3, 3), strides=stride, padding='same', use_bias=False, kernel_regularizer=l2(1e-4))(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)

    # Conv 2
    x = Conv2D(filters, (3, 3), padding='same', use_bias=False, kernel_regularizer=l2(1e-4))(x)
    x = BatchNormalization()(x)

    # Shortcut adjustment
    if downsample or shortcut.shape[-1] != filters:
        shortcut = Conv2D(filters, (1, 1), strides=stride, padding='same', use_bias=False)(shortcut)
        shortcut = BatchNormalization()(shortcut)

    # Residual connection
    x = Add()([x, shortcut])
    x = Activation('relu')(x)

    return x

In [ ]:
# Workflow / Foward Pass Model Functional
inputs = Input(shape=(160, 160, 3))
x = data_augmentation(inputs)

x = Conv2D(32, (3, 3), padding='same', use_bias=False)(x)
x = BatchNormalization()(x)
x = Activation('relu')(x)

x = residual_block(x, 16)
x = residual_block(x, 32, downsample=True)
x = residual_block(x, 64, downsample=True)
x = residual_block(x, 128, downsample=True)

gap = GlobalAveragePooling2D()(x)
gmp = GlobalMaxPooling2D()(x)
x = Concatenate()([gap, gmp])

x = Dense(128, activation='relu')(x)
x = Dropout(0.5)(x)
outputs = Dense(1, activation='sigmoid')(x)

model = Model(inputs, outputs)

model.summary()

### Custom Training & Evaluation Loop dengan tf.GradientTape

In [ ]:
# Mempersiapkan Komponen Training
optimizer = Adam(1e-4)
loss_fn = BinaryCrossentropy(label_smoothing=0.1)

train_acc_metric = tf.keras.metrics.BinaryAccuracy(name='accuracy')
val_acc_metric = tf.keras.metrics.BinaryAccuracy(name='val_accuracy')

train_auc_metric = tf.keras.metrics.AUC(name='auc')
val_auc_metric = tf.keras.metrics.AUC(name='val_auc')

train_loss_metric = tf.keras.metrics.Mean(name='loss')
val_loss_metric = tf.keras.metrics.Mean(name='val_loss')

In [ ]:
# Set Up CheckpointManager
checkpoint = tf.train.Checkpoint(
    model=model,
    optimizer=optimizer
)

checkpoint_manager = tf.train.CheckpointManager(
    checkpoint,
    directory=model_prog_dir,
    max_to_keep=5
)

In [ ]:
# Membuat Fungsi Pelatihan
@tf.function
def train_step(images, labels):
    with tf.GradientTape() as tape:
        predictions = model(images, training=True)
        loss = loss_fn(labels, predictions)
        if model.losses:
            loss += tf.add_n(model.losses)

    gradients = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(gradients, model.trainable_variables))

    train_loss_metric.update_state(loss)
    train_acc_metric.update_state(labels, predictions)
    train_auc_metric.update_state(labels, predictions)

# Membuat Fungsi Validation
@tf.function
def val_step(images, labels):
    predictions = model(images, training=False)
    loss = loss_fn(labels, predictions)
    if model.losses:
        loss += tf.add_n(model.losses)

    val_loss_metric.update_state(loss)
    val_acc_metric.update_state(labels, predictions)
    val_auc_metric.update_state(labels, predictions)

In [ ]:
# Membuat Fungsi Training Loop
def custom_train(model, train_gen, val_gen, epochs=30, initial_epoch=0, lr_factor=0.5, lr_patience=3, min_lr=1e-6, es_patience=5):
    # Hubungkan Callbacks ke Model
    callback_list = tf.keras.callbacks.CallbackList(
        callbacks,
        add_history=True,
        model=model
    )
    callback_list.set_params({
        "epochs": epochs,
        "steps": len(train_gen),
        "verbose": 1
    })

    # Mulai Training
    callback_list.on_train_begin()
    logs = {}
    best_val_loss = float("inf")

    lr_counter = 0
    es_counter = 0

    for epoch in range(initial_epoch, epochs):
        print(f"\nEpoch {epoch+1}/{epochs}")

        # Reset metrics di awal setiap epoch
        for metric in [
            train_loss_metric, train_acc_metric, train_auc_metric,
            val_loss_metric, val_acc_metric, val_auc_metric
        ]:
            metric.reset_state()

        # Callback awal epoch
        callback_list.on_epoch_begin(epoch)

        # Training loop
        with tqdm(
            train_gen,
            total=len(train_gen),
            desc=f"Training Epoch {epoch+1}"
        ) as pbar:

            for images, labels in pbar:
                train_step(images, labels)

                pbar.set_postfix({
                    "loss": f"{float(train_loss_metric.result()):.4f}",
                    "acc": f"{float(train_acc_metric.result()):.4f}"
                })

        # Validation loop
        for images, labels in tqdm(
            val_gen,
            total=len(val_gen),
            desc=f"Validation Epoch {epoch+1}"
        ):
            val_step(images, labels)

        # Ambil hasil metric
        train_loss = train_loss_metric.result()
        train_acc = train_acc_metric.result()
        train_auc = train_auc_metric.result()

        val_loss = val_loss_metric.result()
        val_acc = val_acc_metric.result()
        val_auc = val_auc_metric.result()

        current_lr = float(tf.keras.backend.get_value(optimizer.learning_rate))

        # Tensorboard Logging
        with train_writer.as_default():
            tf.summary.scalar("loss", train_loss, step=epoch)
            tf.summary.scalar("accuracy", train_acc, step=epoch)
            tf.summary.scalar("auc", train_auc, step=epoch)
            train_writer.flush()

        with val_writer.as_default():
            tf.summary.scalar("loss", val_loss, step=epoch)
            tf.summary.scalar("accuracy", val_acc, step=epoch)
            tf.summary.scalar("auc", val_auc, step=epoch)
            val_writer.flush()

        # Tampilkan hasil
        print(
            f"loss          : {train_loss:.4f}\n"
            f"accuracy      : {train_acc:.4f}\n"
            f"auc           : {train_auc:.4f}\n"
            f"val_loss      : {val_loss:.4f}\n"
            f"val_accuracy  : {val_acc:.4f}\n"
            f"val_auc       : {val_auc:.4f}\n"
            f"learning_rate : {current_lr:.6f}"
        )

        # Logs untuk callback
        logs = {
            "loss": float(train_loss.numpy()),
            "accuracy": float(train_acc.numpy()),
            "auc": float(train_auc.numpy()),
            "val_loss": float(val_loss.numpy()),
            "val_accuracy": float(val_acc.numpy()),
            "val_auc": float(val_auc.numpy()),
            "learning_rate": current_lr
        }

        # Callback akhir epoch
        callback_list.on_epoch_end(epoch, logs)

        # Save Checkpoint
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            lr_counter = 0
            es_counter = 0
            checkpoint_manager.save()
            print("Checkpoint disimpan karena val_loss menurun")
        else:
            lr_counter += 1
            es_counter += 1

            # ReduceLROnPlateau
            if lr_counter >= lr_patience:
                old_lr = float(optimizer.learning_rate.numpy())
                new_lr = max(old_lr * lr_factor, min_lr)

                optimizer.learning_rate.assign(new_lr)

                print(f"Learning rate diturunkan dari {old_lr:.6f} menjadi {new_lr:.6f}")
                lr_counter = 0

            # EarlyStopping
            if es_counter >= es_patience:
                print("Training dihentikan oleh early stopping")
                break

    # Callback akhir training
    callback_list.on_train_end(logs)

### Training Model

In [ ]:
# train_batches = tf.data.experimental.cardinality(train_gen).numpy()
# train_gen_small = train_gen.take(int(train_batches * 0.10))

In [ ]:
# Menjalankan Training
custom_train(
    model=model,
    train_gen=train_gen,
    val_gen=val_gen,
    epochs=30,
    lr_factor=0.5,
    lr_patience=3,
    min_lr=1e-6,
    es_patience=5
)

In [ ]:
# Cek apakah ada checkpoint yang tersedia
if checkpoint_manager.latest_checkpoint:
    checkpoint.restore(checkpoint_manager.latest_checkpoint)
    print("Checkpoint berhasil dimuat dari:", checkpoint_manager.latest_checkpoint)
else:
    print("Tidak ditemukan checkpoint, training akan dimulai dari awal.")

In [ ]:
# # Lanjutkan Training
# custom_train(
#     model=model,
#     train_gen=train_gen,
#     val_gen=val_gen,
#     epochs=30,
#     initial_epoch=24,
#     lr_factor=0.5,
#     lr_patience=3,
#     min_lr=1e-6,
#     es_patience=5
# )

## Evaluasi dan Visualisasi

### Plotting Akurasi dan Loss

In [ ]:
# Baca seluruh riwayat training
df_log = pd.read_csv(f"{model_prog_dir}/training_log.csv")

epochs = df_log['epoch'] + 1
acc = df_log['accuracy']
val_acc = df_log['val_accuracy']
loss = df_log['loss']
val_loss = df_log['val_loss']

# Figure Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Plot Akurasi
ax1.plot(epochs, acc, 'r', label='train')
ax1.plot(epochs, val_acc, 'b', label='val')
ax1.set_title('Training and Validation Accuracy')
ax1.set_xlabel('epoch')
ax1.set_ylabel('accuracy')
ax1.legend(loc='upper left')
ax1.grid(True)

# Plot Loss
ax2.plot(epochs, loss, 'r', label='train')
ax2.plot(epochs, val_loss, 'b', label='val')
ax2.set_title('Training and Validation Loss')
ax2.set_xlabel('epoch')
ax2.set_ylabel('loss')
ax2.legend(loc='upper left')

plt.tight_layout()
plt.show()

### Evaluasi Prediksi Data Train dan Test


In [ ]:
# Custom get target / y_true
def get_target(gen_data):
    ytrue = []
    for images, labels in gen_data:
        ytrue.extend(labels.numpy())
    ytrue = np.array(ytrue).flatten()
    return ytrue

In [ ]:
# Prediksi Data Train
train_pred = model.predict(train_pred_gen)
train_pred = (train_pred > 0.5).astype(int)
train_pred = train_pred.flatten()

# Menghitung akurasi prediksi
train_true = get_target(train_pred_gen)
train_acc = np.mean(train_pred == train_true)
print("Train Accuracy:", train_acc)

In [ ]:
# Prediksi Data Validation
val_pred = model.predict(val_gen).flatten()

# Menghitung akurasi prediksi
val_true = get_target(val_gen)
val_pred1 = (val_pred > 0.5).astype(int)
val_acc1 = np.mean(val_pred1 == val_true)
print("Validation Accuracy (Treshold = 0.5):", val_acc1)

# Mencari best treshold
fpr, tpr, thresholds = roc_curve(val_true, val_pred)
gmeans = np.sqrt(tpr * (1 - fpr))
ix = np.argmax(gmeans)
best_threshold = thresholds[ix]
print(f"Best Treshold: {best_threshold:.4f}")

val_pred2 = (val_pred > best_threshold).astype(int)
val_acc2 = np.mean(val_pred2 == val_true)
print(f"Validation Accuracy (Best Treshold) : {val_acc2 * 100:.2f}%")

In [ ]:
# Prediksi Data Test
test_pred = model.predict(test_gen).flatten()
test_pred = (test_pred > 0.5).astype(int)
test_pred = test_pred.flatten()

# Menghitung akurasi prediksi
test_true = get_target(test_gen)
test_pred1 = (test_pred > 0.5).astype(int)
test_acc1 = np.mean(test_pred1 == test_true)
print("Test Accuracy (Treshold = 0.5):", test_acc1)

test_pred2 = (test_pred > best_threshold).astype(int)
test_acc2 = np.mean(test_pred2 == test_true)
print("Test Accuracy (Best Treshold):", test_acc2)

In [ ]:
# MAE
test_prob = model.predict(test_gen).flatten()
test_target = get_target(test_gen)

mae = mean_absolute_error(test_target, test_prob)
print("MAE (Test Dataset):", mae)

### Confusion Matrix & Classification Report

In [ ]:
# Label Confusion Matrix
label_class = train_class
actual_labels = [f"Actual {name}" for name in label_class]
predicted_labels = [f"Predicted {name}" for name in label_class]

# Custom Confusion Matrix Function
def plot_confusion_matrix(preds, gen_data, title="Confusion Matrix"):
    # mengubah array menjadi bentuk 1 dimensi (1D)
    preds = np.ravel(preds)

    cm = confusion_matrix(
        get_target(gen_data),
        preds,
        labels=range(len(label_class))
    )

    cm_df = pd.DataFrame(
        cm,
        index=actual_labels,
        columns=predicted_labels
    )

    plt.figure(figsize=(6,5))
    sns.heatmap(cm_df, annot=True, fmt="d")
    plt.title(title)
    plt.show()

# Custom Classification Report Function
def print_classification_report(preds, gen_data):
    print("\n")
    print(classification_report(
        y_true=get_target(gen_data),
        y_pred=preds,
        target_names=label_class,
        digits=4
    ))

In [ ]:
# Confusion Matrix & Classification Report Train
plot_confusion_matrix(train_pred, train_pred_gen, title="Train Confusion Matrix")
print_classification_report(train_pred, train_pred_gen)

In [ ]:
# Confusion Matrix & Classification Report Test
plot_confusion_matrix(test_pred, test_gen, title="Test Confusion Matrix")
print_classification_report(test_pred, test_gen)

## Simpan Model

In [ ]:
# Tentukan direktori utama penyimpanan
save_dir = f"{MODEL_DIR}/final_model"
os.makedirs(save_dir, exist_ok=True)

In [ ]:
# Simpan dalam bentuk keras dan saved_model
model.save(f"{save_dir}/model.keras")
print(f"✅ Selesai konversi ke keras: {save_dir}/model.keras")

model.export(f"{save_dir}/saved_model")
print(f"✅ Selesai konversi ke SavedModel: {save_dir}/saved_model/")

In [ ]:
# Simpan dalam bentuk TFLite
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

tflite_path = f"{save_dir}/model.tflite"
with open(tflite_path, "wb") as f:
    f.write(tflite_model)
print(f"✅ Selesai konversi ke TF-Lite: {tflite_path}")

In [ ]:
# Simpan label.txt
labels_path = f"{save_dir}/labels.txt"

with open(labels_path, "w") as f:
    for name in train_class:
        f.write(f"{name}\n")

print(f"✅ Selesai simpan label: {labels_path}")
print("Daftar kelas yang tersimpan:", train_class)

## Inferensi Model

### Load & Preprocessing Data

In [ ]:
# Preprocessing Gambar
def load_inference_data(infer_dir, image_size=(160, 160), batch_size=64):
    # Load dataset
    infer_gen = tf.keras.utils.image_dataset_from_directory(
        infer_dir,
        image_size=image_size,
        batch_size=batch_size,
        label_mode=None,
        shuffle=False
    )
    # Simpan file paths
    file_paths = infer_gen.file_paths
    # Rescaling
    scaler = tf.keras.layers.Rescaling(1./255)
    infer_gen = infer_gen.map(
        lambda x: scaler(x),
        num_parallel_calls=tf.data.AUTOTUNE
    )
    return infer_gen, file_paths

In [ ]:
infer_dir = f"{MODEL_DIR}/data_inference"
os.makedirs(infer_dir, exist_ok=True)


sample_images_to_copy = 3 # Jumlah angka untuk di copy tiap kelas

# Ambil sampel gambar dari kelas test
for class_name in os.listdir(test_dir):
    class_path = os.path.join(test_dir, class_name)
    if os.path.isdir(class_path):
        images_in_class = [f for f in os.listdir(class_path) if f.lower().endswith(('.png', '.jpg', '.jpeg', '.gif', '.bmp'))]
        random.shuffle(images_in_class)
        for i, image_name in enumerate(images_in_class):
            if i >= sample_images_to_copy:
                break
            source_path = os.path.join(class_path, image_name)
            destination_path = os.path.join(infer_dir, f"{class_name}_{image_name}") # Rename to avoid conflicts and keep class info
            shutil.copy(source_path, destination_path)

print(f"Menyalin {sample_images_to_copy * len(os.listdir(test_dir))} sampel gambar ke {infer_dir}")

infer_gen, file_paths = load_inference_data(
    infer_dir=infer_dir,
    image_size=(160, 160),
    batch_size=6
)

In [ ]:
# Lihat gambar inference
for images in infer_gen.take(1):
    plt.figure(figsize=(12, 12))
    for i in range(len(images)):
        plt.subplot(2, 3, i + 1)
        img = images[i].numpy()
        img = np.clip(img, 0, 1)
        plt.imshow(img)
        plt.title(
            os.path.basename(file_paths[i]),
            fontsize=10
        )
        plt.axis("off")
    plt.tight_layout()
    plt.show()

### Load Model & Predict

In [ ]:
# Load Model
model = tf.keras.models.load_model(
    f"{MODEL_DIR}/final_model/model.keras"
)

print("Model berhasil dimuat")
model.summary()

In [ ]:
# Load Label
labels_path = f"{MODEL_DIR}/final_model/labels.txt"

with open(labels_path, "r") as f:
    class_names = [
        line.strip()
        for line in f.readlines()
    ]

print(class_names)

In [ ]:
# Predict seluruh data inference
predictions = model.predict(infer_gen)
pred_labels = (predictions >= 0.5).astype(int)

# Tampilkan hasil prediksi
for path, prob, label in zip(file_paths, predictions, pred_labels):
    # Tampilkan gambar
    img = tf.keras.utils.load_img(
        path,
        target_size=(160,160)
    )
    img = tf.keras.utils.img_to_array(img).astype("uint8")
    plt.figure(figsize=(4,4))
    plt.imshow(img)
    plt.axis("off")
    plt.show()

    # Informasi prediksi
    print(f"File       : {os.path.basename(path)}")
    print(f"Prediksi   : {class_names[label[0]]}")
    print(f"Probability: {prob[0]:.4f}")
    print("-" * 40)

In [ ]:
!rm -rf data